In [1]:
import os
import sys
import glob
import numpy as np
from tqdm import trange
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.convolution import convolve, Gaussian1DKernel
import astropy.units as u
import astropy.coordinates as coord
import matplotlib
import matplotlib.pyplot as plt
from astropy.table import Column
from tqdm import trange
import pandas as pd
import pickle
import fitsio
from astropy.table import Table, vstack
from astropy import units as u
from astropy.coordinates import SkyCoord
from easyquery import Query, QueryMaker
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from matplotlib.colors import ListedColormap, BoundaryNorm
import h5py
from astropy.cosmology import Planck18

rootdir = '/global/u1/v/virajvm/'
sys.path.append(os.path.join(rootdir, 'DESI2_LOWZ/desi_dwarfs/code'))

sys.path.append(os.path.join(rootdir, 'DESI2_LOWZ/desi_dwarfs/code/nebular_stuff'))

from desi_lowz_funcs import make_subplots, match_c_to_catalog, print_radecs
from desi_lowz_funcs import calc_normalized_dist
from desi_lowz_funcs import find_objects_nearby
# from construct_dwarf_galaxy_catalogs import process_sga_matches
from catalog_paper_plots import make_bar_pie


import numpy as np
import h5py
from astropy.table import Table
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

# Key emission lines for reference (rest-frame wavelengths in Angstroms)
emission_lines = {
    r"[OII] $\lambda$3727": 3727.0,
    r"H$\beta$": 4861.0,
    r"[OIII] $\lambda$4959": 4959.0,
    r"[OIII] $\lambda$5007": 5007.0,
    r"H$\alpha$": 6563.0,
    r"[NII] $\lambda$6584": 6584.0,
    r"[SII] $\lambda$6717": 6717.0,
    r"[SII] $\lambda$6731": 6731.0,
}


%load_ext autoreload
%autoreload 2


In [3]:
bgs_cat = Table.read("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/iron_photometry/iron_BGS_BRIGHT_shreds_catalog_w_aper_mags.fits")

sga_cat = Table.read("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/iron_photometry/iron_SGA_sga_catalog_w_aper_mags.fits")


In [8]:
bgs_cat_f = bgs_cat["RA","DEC", "IMAGE_SIZE_PIX", "FILE_PATH", "TARGETID", "BRICKNAME", "Z", "MASKBITS", "is_south" , "APER_PARAMS_ISOLATE", "APER_CEN_XY_PIX_ISOLATE", "COG_MAG_G_ISOLATE", "COG_MAG_R_ISOLATE", "COG_MAG_Z_ISOLATE"]
sga_cat_f = sga_cat["RA","DEC", "IMAGE_SIZE_PIX", "FILE_PATH", "TARGETID", "BRICKNAME", "Z", "MASKBITS", "is_south" , "APER_PARAMS_ISOLATE", "APER_CEN_XY_PIX_ISOLATE", "COG_MAG_G_ISOLATE", "COG_MAG_R_ISOLATE", "COG_MAG_Z_ISOLATE"]


In [9]:
eg_tgids = np.array([39627426581450605, 39627497389687574, 39627583502946875, 39627625336934247, 39627687479742998, 39627764382305600])



In [11]:
for tgidi in eg_tgids:
    bgs_cat_i = bgs_cat_f[bgs_cat_f["TARGETID"] == tgidi]
    sga_cat_i = sga_cat_f[sga_cat_f["TARGETID"] == tgidi]

    if len(bgs_cat_i) > 0:
        print(bgs_cat_i["TARGETID"][0])
        print(bgs_cat_i["FILE_PATH"][0])
        
    elif len(sga_cat_f) > 0:
        print(sga_cat_i["TARGETID"][0])
        print(sga_cat_i["FILE_PATH"][0])
    else:
        print(f"MISSING TGID : {tgidi}")

    print("---")

39627426581450605
/pscratch/sd/v/virajvm/redo_photometry_plots/all_deshreds/south/sweep-020m015-030m010/0205m150/BGS_BRIGHT_tgid_39627426581450605
---
39627497389687574
/pscratch/sd/v/virajvm/redo_photometry_plots/all_deshreds/south/sweep-030m015-040m010/0386m120/BGS_BRIGHT_tgid_39627497389687574
---
39627583502946875
/pscratch/sd/v/virajvm/redo_photometry_plots/all_deshreds/south/sweep-210m010-220m005/2106m085/BGS_BRIGHT_tgid_39627583502946875
---
39627625336934247
/pscratch/sd/v/virajvm/redo_photometry_plots/all_deshreds/south/sweep-200m010-210m005/2037m067/BGS_BRIGHT_tgid_39627625336934247
---
39627687479742998
/pscratch/sd/v/virajvm/redo_photometry_plots/all_deshreds/south/sweep-320m005-330p000/3215m042/BGS_BRIGHT_tgid_39627687479742998
---
39627764382305600
/pscratch/sd/v/virajvm/redo_photometry_plots/all_deshreds/south/sweep-220m005-230p000/2273m010/BGS_BRIGHT_tgid_39627764382305600
---


In [12]:
import os, sys, shutil
import numpy as np
from astropy.table import vstack

# ---------------- config ----------------
REPO_ROOT = "/global/u1/v/virajvm/DESI2_LOWZ/desi_dwarfs"     # canonical repo root
OUT_DIR   = "/pscratch/sd/v/virajvm/scarlet_local_test"        # bundle to download
sys.path.insert(0, os.path.join(REPO_ROOT, "code"))
import cutout_store
CUTOUTS_DIR = cutout_store.get_store_dir()                     # the NERSC image store

# the exact columns the fitter reads (your list; LOGM deliberately omitted)
COLS = ["RA", "DEC", "IMAGE_SIZE_PIX", "FILE_PATH", "TARGETID", "BRICKNAME",
        "Z", "MASKBITS", "is_south", "APER_PARAMS_ISOLATE", "APER_CEN_XY_PIX_ISOLATE",
        "COG_MAG_G_ISOLATE", "COG_MAG_R_ISOLATE", "COG_MAG_Z_ISOLATE"]

# per-object FILE_PATH products to copy (source cat required; .npy optional-but-wanted)
PER_OBJECT_FILES = ["source_cat_f_more.fits", "source_cat_f.fits",
                    "segment_map_v2.npy", "star_mask.npy",
                    "fiber_pix_pos.npy", "noise_per_band_rms.npy"]

# ---------------- setup ----------------
objects_dir = os.path.join(OUT_DIR, "objects")
images_dir  = os.path.join(OUT_DIR, "images")
os.makedirs(objects_dir, exist_ok=True)
os.makedirs(images_dir,  exist_ok=True)

bgs_sel = bgs_cat[[c for c in COLS if c in bgs_cat.colnames]]
sga_sel = sga_cat[[c for c in COLS if c in sga_cat.colnames]]

slice_rows = []
for tgidi in eg_tgids:
    sub = None
    for cat in (bgs_sel, sga_sel):
        m = cat[cat["TARGETID"] == tgidi]
        if len(m) > 0:
            sub = m[:1]
            break
    if sub is None:
        print(f"MISSING TGID : {tgidi}"); continue

    tgid      = int(sub["TARGETID"][0])
    brick     = str(sub["BRICKNAME"][0])
    file_path = str(sub["FILE_PATH"][0])

    # 1) per-object files  ->  objects/{tgid}/
    dst = os.path.join(objects_dir, str(tgid))
    os.makedirs(dst, exist_ok=True)
    got_srccat = False
    for fn in PER_OBJECT_FILES:
        src = os.path.join(file_path, fn)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(dst, fn))
            if fn.startswith("source_cat_f"):
                got_srccat = True
    if not got_srccat:
        print(f"  !! {tgid}: no source_cat_f_more/f.fits in {file_path} (fit will skip)")

    # 2) cutout -> native mini-shard images/{brick[:3]}/{brick}.h5
    try:
        cut = cutout_store.read_cutout(CUTOUTS_DIR, brick, tgid, size=None)  # full stored cube
        rec = {
            "targetid": tgid,
            "image":   cut["image"],
            "invvar":  cut.get("invvar"),
            "mask":    cut.get("mask"),
            "header":  cut["header"],
            "ra":      float(cut.get("ra",  sub["RA"][0])),
            "dec":     float(cut.get("dec", sub["DEC"][0])),
            "box_size": int(cut["image"].shape[-1]),
            "fetch_method": str(cut.get("fetch_method", "copied")),
            "layer":        str(cut.get("layer", "ls-dr9")),
        }
        cutout_store.write_cutouts_batch(images_dir, brick, [rec])
        iv_ok = rec["invvar"] is not None
        print(f"OK  {tgid}  brick={brick}  {rec['box_size']}px  invvar={iv_ok}"
              + ("" if iv_ok else "   !! NO INVVAR -- fit needs it"))
    except Exception as e:
        print(f"  !! {tgid}: cutout extract failed: {e!r}")

    slice_rows.append(sub)

# 3) catalog slice
if slice_rows:
    slice_tbl = vstack(slice_rows)
    slice_tbl.write(os.path.join(OUT_DIR, "test_cat_slice.fits"), overwrite=True)
    print(f"\nWrote catalog slice: {len(slice_tbl)} rows")

# size report
tot = sum(os.path.getsize(os.path.join(r, f))
          for r, _, fs in os.walk(OUT_DIR) for f in fs)
print(f"Bundle ready: {OUT_DIR}  ({tot/1e6:.1f} MB)")
print(f"Download with:  rsync -avz perlmutter:{OUT_DIR} ~/Downloads/")

OK  39627426581450605  brick=0205m150  350px  invvar=True
OK  39627497389687574  brick=0386m120  350px  invvar=True
OK  39627583502946875  brick=2106m085  350px  invvar=True
OK  39627625336934247  brick=2037m067  350px  invvar=True
OK  39627687479742998  brick=3215m042  350px  invvar=True
OK  39627764382305600  brick=2273m010  1094px  invvar=True

Wrote catalog slice: 6 rows
Bundle ready: /pscratch/sd/v/virajvm/scarlet_local_test  (46.9 MB)
Download with:  rsync -avz perlmutter:/pscratch/sd/v/virajvm/scarlet_local_test ~/Downloads/
